# Qwen3-32B 회의 요약 Fine-tuning (극한 설정)

## 메모리 최적화
- **MAX_SEQ_LENGTH**: 512
- **LoRA**: r=8
- **Batch**: 1 x 16
- **텍스트 제한**: 400자

## 환경
- **GPU**: SSAFY L40S 46GB
- **모델**: Qwen3-32B (4-bit)
- **최적화**: Unsloth + LoRA + bf16

In [ ]:
# ===== 0. 환경 설정 =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
import json
import numpy as np
from sklearn.model_selection import train_test_split

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

---
## 1. 데이터 로드 및 분할 (70/15/15)

In [ ]:
# 데이터 로드
with open("training_data_final.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"전체 데이터: {len(raw_data)}개")

# ===== 핵심: Train/Val/Test 분할 =====
train_data, temp_data = train_test_split(raw_data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"\n[데이터 분할]")
print(f"  Train:      {len(train_data):>5}개 (70%)")
print(f"  Validation: {len(val_data):>5}개 (15%)")
print(f"  Test:       {len(test_data):>5}개 (15%)")

# Test 데이터 저장 (나중에 최종 평가용)
with open("test_data.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)
print(f"\nTest 데이터 저장: test_data.json")

---
## 2. 모델 로드

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-32B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512  # 극단적으로 줄임 (메모리 절약)

print(f"모델 로딩: {MODEL_NAME}")
print("4-bit 양자화로 로딩 중...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"\n모델 로드 완료!")
print(f"VRAM 사용: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

---
## 3. [BEFORE] 파인튜닝 전 성능 테스트

In [ ]:
# 테스트 샘플 (고정 - Before/After 비교용)
TEST_SAMPLES = [
    """김개발: 오늘 스프린트 회의 시작합니다. 진행 상황 공유해주세요.
이디자인: 메인 페이지 UI 70% 완료했습니다. 내일 마무리 예정이에요.
박백엔드: 로그인 API 완료했고 PR 올렸습니다. 리뷰 부탁드려요.
김개발: 좋습니다. 블로커 있으신 분?
이디자인: 아이콘 에셋이 아직 안 나왔어요. 디자인팀에 요청해야 할 것 같습니다.
김개발: 제가 오늘 중으로 요청할게요. 다음 스프린트 목표 논의하죠.
박백엔드: 결제 모듈 연동 시작하면 좋겠습니다.""",
    
    """팀장: CI/CD 파이프라인 상태 공유해주세요.
시니어: Jenkins 빌드 자동화 완료했습니다. develop 브랜치 푸시하면 자동 빌드됩니다.
주니어: 스테이징 배포는 되는데, 프로덕션은 아직 수동이에요.
팀장: 프로덕션도 자동화하면 좋겠네요.
시니어: 이번 주 내로 ArgoCD 연동해서 구축하겠습니다.
주니어: 저는 Grafana 모니터링 대시보드 설정 마무리할게요.
팀장: 다음 주 수요일까지 완료 부탁드립니다."""
]

PROMPT_TEMPLATE = """반드시 한국어로만 답변하세요. 영어로 생각하지 마세요.

다음 회의 내용을 분석하여 정리해주세요.

회의 내용:
{content}

다음 형식으로 응답:
## 요약
[2-3문장 핵심 요약]

## 액션 아이템
- [담당자]: [할 일]

## 키워드
[쉼표로 구분]

응답:"""

def generate_summary(model, tokenizer, text, max_new_tokens=512):
    prompt = PROMPT_TEMPLATE.format(content=text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,  # 낮춰서 안정적인 출력
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("응답:")[-1].strip()

In [ ]:
# Before 결과 저장
FastLanguageModel.for_inference(model)

print("=" * 60)
print("[BEFORE] 파인튜닝 전 성능")
print("=" * 60)

before_results = []
for i, sample in enumerate(TEST_SAMPLES):
    print(f"\n--- 샘플 {i+1} ---")
    result = generate_summary(model, tokenizer, sample)
    before_results.append(result)
    print(result[:500] if len(result) > 500 else result)

---
## 4. LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # 메모리 절약
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA 설정 완료 (r=8, 32B 극한 모드)")
model.print_trainable_parameters()

---
## 5. 데이터셋 준비

In [ ]:
from datasets import Dataset

def format_for_training(item):
    """학습용 포맷 변환"""
    text = item.get("text", "")[:400]  # 400자로 더 줄임 (512 토큰 이내)
    
    formatted = f"""내용: {text}

정리: 개발 관련 내용입니다."""
    
    return {"text": formatted}

# Train/Val 데이터셋 생성
train_dataset = Dataset.from_list([format_for_training(item) for item in train_data])
val_dataset = Dataset.from_list([format_for_training(item) for item in val_data])

# 토큰 길이 확인
sample_tokens = tokenizer(train_dataset[0]['text'])
print(f"Train 데이터셋: {len(train_dataset)}개")
print(f"샘플 토큰 수: {len(sample_tokens['input_ids'])}")
print(f"\n[샘플]\n{train_dataset[0]['text'][:200]}...")

---
## 6. Trainer 설정 (32B 극한 메모리 모드)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen3-meeting-summary-lora",
    
    # === 32B 극한 설정 ===
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    
    # === 최적화 ===
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=False,
    bf16=True,
    
    # === 평가/저장 비활성화 (메모리 절약) ===
    eval_strategy="no",
    save_strategy="no",
    
    # === 로깅 ===
    logging_steps=5,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("[Trainer 설정 - 32B 극한 모드]")
print(f"  - MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"  - Batch: 1 x 16")
print(f"  - Epochs: 1")

---
## 7. 학습 실행

In [ ]:
import gc

# 메모리 정리
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("학습 시작")
print("=" * 60)
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print("\nValidation Loss가 3회 연속 개선되지 않으면 자동 종료됩니다.\n")

train_result = trainer.train()

print("\n" + "=" * 60)
print("학습 완료!")
print("=" * 60)
print(f"Total steps: {train_result.global_step}")
print(f"Final train loss: {train_result.training_loss:.4f}")

---
## 8. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

# Train Loss 추출
train_loss = [(x['step'], x['loss']) for x in logs if 'loss' in x]

plt.figure(figsize=(10, 5))

if train_loss:
    steps, losses = zip(*train_loss)
    plt.plot(steps, losses, label='Train Loss', marker='o', alpha=0.7)
    
    # 최저점 표시
    min_idx = np.argmin(losses)
    plt.scatter([steps[min_idx]], [losses[min_idx]], color='red', s=100, zorder=5)
    plt.annotate(f'Min: {losses[min_idx]:.4f}', 
                 xy=(steps[min_idx], losses[min_idx]),
                 xytext=(10, 10), textcoords='offset points')

plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150)
plt.show()

print(f"\n[학습 결과]")
if train_loss:
    print(f"  Final Loss: {losses[-1]:.4f}")
    print(f"  Min Loss: {min(losses):.4f} (step {steps[np.argmin(losses)]})")

---
## 9. [AFTER] 파인튜닝 후 성능 테스트

In [ ]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("[AFTER] 파인튜닝 후 성능")
print("=" * 60)

after_results = []
for i, sample in enumerate(TEST_SAMPLES):
    print(f"\n--- 샘플 {i+1} ---")
    result = generate_summary(model, tokenizer, sample)
    after_results.append(result)
    print(result[:500] if len(result) > 500 else result)

---
## 10. Before/After 비교

In [ ]:
print("=" * 70)
print("[COMPARISON] Before vs After")
print("=" * 70)

for i in range(len(TEST_SAMPLES)):
    print(f"\n{'='*70}")
    print(f"샘플 {i+1}")
    print(f"{'='*70}")
    print(f"\n[INPUT]")
    print(TEST_SAMPLES[i][:150] + "...")
    print(f"\n[BEFORE - 파인튜닝 전]")
    print(before_results[i][:300] + "..." if len(before_results[i]) > 300 else before_results[i])
    print(f"\n[AFTER - 파인튜닝 후]")
    print(after_results[i][:300] + "..." if len(after_results[i]) > 300 else after_results[i])

---
## 11. GPT-4 비교 평가 (선택)

In [ ]:
# ===== OpenAI API 키 설정 (없으면 스킵) =====
OPENAI_API_KEY = ""  # 여기에 입력, 없으면 빈 문자열

gpt4_results = None

if OPENAI_API_KEY:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    def get_gpt4_summary(text):
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "회의 내용을 요약해주세요. 요약, 액션아이템, 키워드를 포함해주세요."},
                {"role": "user", "content": text}
            ],
            temperature=0.7,
        )
        return response.choices[0].message.content
    
    print("GPT-4 요약 생성 중...")
    gpt4_results = [get_gpt4_summary(sample) for sample in TEST_SAMPLES]
    
    for i, result in enumerate(gpt4_results):
        print(f"\n--- GPT-4 샘플 {i+1} ---")
        print(result[:500])
else:
    print("OpenAI API 키가 없습니다. GPT-4 비교를 건너뜁니다.")
    print("\n[대안] 나중에 API 키 추가 후 이 셀만 다시 실행하면 됩니다.")

---
## 12. 유사도 측정 (정량 평가)

In [ ]:
# sentence-transformers 설치 필요: pip install sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    
    # 한국어 임베딩 모델
    embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
    
    def calc_similarity(text1, text2):
        emb1 = embed_model.encode([text1])
        emb2 = embed_model.encode([text2])
        return cosine_similarity(emb1, emb2)[0][0]
    
    print("=" * 60)
    print("[유사도 측정 결과]")
    print("=" * 60)
    
    for i in range(len(TEST_SAMPLES)):
        print(f"\n샘플 {i+1}:")
        
        # Before vs After
        ba_sim = calc_similarity(before_results[i], after_results[i])
        print(f"  Before ↔ After: {ba_sim:.1%}")
        
        # GPT-4 비교 (있을 경우)
        if gpt4_results:
            bg_sim = calc_similarity(before_results[i], gpt4_results[i])
            ag_sim = calc_similarity(after_results[i], gpt4_results[i])
            print(f"  Before ↔ GPT-4: {bg_sim:.1%}")
            print(f"  After ↔ GPT-4:  {ag_sim:.1%}  {'↑ 개선!' if ag_sim > bg_sim else ''}")
    
    # 평균
    if gpt4_results:
        avg_before = np.mean([calc_similarity(before_results[i], gpt4_results[i]) for i in range(len(TEST_SAMPLES))])
        avg_after = np.mean([calc_similarity(after_results[i], gpt4_results[i]) for i in range(len(TEST_SAMPLES))])
        
        print(f"\n{'='*60}")
        print(f"[평균 GPT-4 유사도]")
        print(f"  Before: {avg_before:.1%}")
        print(f"  After:  {avg_after:.1%}")
        print(f"  개선:   +{(avg_after - avg_before):.1%}")
        
except ImportError:
    print("sentence-transformers 없음. 유사도 측정 스킵.")
    print("설치: pip install sentence-transformers")

---
## 13. 모델 저장

In [ ]:
# LoRA 어댑터 저장
SAVE_PATH = "./qwen3-meeting-summary-lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"모델 저장 완료: {SAVE_PATH}")
print(f"\n[저장된 파일]")
import os
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

---
## 14. 최종 리포트

In [ ]:
print("\n" + "=" * 60)
print("최종 학습 리포트")
print("=" * 60)

print(f"\n[모델]")
print(f"  Base: {MODEL_NAME}")
print(f"  Method: LoRA (r=8, alpha=8)")
print(f"  Quantization: 4-bit")
print(f"  Max Seq Length: {MAX_SEQ_LENGTH}")

print(f"\n[데이터]")
print(f"  Train: {len(train_data)}개 (70%)")
print(f"  Test:  {len(test_data)}개 (15%)")

print(f"\n[학습]")
print(f"  Epochs: 1")
print(f"  Steps: {train_result.global_step}")
print(f"  Train Loss: {train_result.training_loss:.4f}")
print(f"  Batch: 1 x 16 (gradient accum)")

print(f"\n[저장]")
print(f"  모델: {SAVE_PATH}")

print("\n" + "=" * 60)
print("완료! 서버에서 다운로드 후 Modal 배포 진행")
print("=" * 60)